In [ ]:
import openeo

connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
print(connection)

In [ ]:
import geopandas as gpd
import Functions
import os
import pandas as pd

app_path = Functions.get_input_path() / 'App'
input_path = app_path / 'Data' / 'Ground_Campaign'
ndvi = app_path / 'Documents' / 'csv' / 'NDVI'

shape_path = input_path / 'Flevoland_data' / 'Data_25_fields' / 'Flevoland-fields-Shapefiles' / 'Feloveland-fields-Shapefiles'
shapes = list(shape_path.glob('*.shp'))

for i in shapes:
    try:
        # Load and validate shapefile
        gdf = gpd.read_file(i)

        if gdf.empty or gdf.geometry.isnull().all():
            print(f"Warning: File {i} is empty. Skipping.")
            continue
            
        # Standardize CRS to WGS84
        if gdf.crs != "EPSG:4326":
            gdf = gdf.to_crs("EPSG:4326")
            
        spatial_extent = gdf.union_all().__geo_interface__
        
        # Initialize openEO data cube
        datacube = connection.load_collection(
            "SENTINEL2_L2A",
            spatial_extent=spatial_extent,
            temporal_extent=["2017-01-03", "2017-12-31"],
            bands=["B04", "B08"],
            properties={"eo:cloud_cover": lambda c: c <= 10}
        )

        # Compute NDVI and extract spatial mean
        ndvi_cube = datacube.ndvi(nir="B08", red="B04")
        stats_cube = ndvi_cube.aggregate_spatial(
            geometries=spatial_extent,
            reducer="mean"
        )
        dati_grafico = stats_cube.execute()

        # Format retrieved data
        records = []
        for data_str, valori in dati_grafico.items():
            valore_ndvi = valori[0][0] if valori else None 
            records.append({
                "Data": pd.to_datetime(data_str).date(),
                "NDVI_Medio": valore_ndvi
            })

        # Process and filter data
        df = pd.DataFrame(records)
        df = df.sort_values(by="Data").reset_index(drop=True)
        df.dropna(inplace=True)
        df = df[df['NDVI_Medio'] > 0.1].dropna()

        # Save to CSV
        df.to_csv(app_path / 'Documents' / 'csv' / 'NDVI' / f'{os.path.basename(i)}.csv', index=False) 

        print(f"File {os.path.basename(i)} processed successfully on openEO.")
        
    except Exception as e:
        # Error handling for individual file processing
        print(f"ERROR: Could not process {os.path.basename(i)}.")
        print(f"Details: {e}")
        continue

In [ ]:
records = []

for data_str, valori in dati_grafico.items():

    valore_ndvi = valori[0][0] if valori else None 
    
    records.append({
        "Data": pd.to_datetime(data_str).date(), # Converte in formato data (AAAA-MM-GG)
        "NDVI_Medio": valore_ndvi
    })

df = pd.DataFrame(records)

df = df.sort_values(by="Data").reset_index(drop=True)

df.dropna(inplace=True)
df = df[df['NDVI_Medio']>0.1].dropna()

In [ ]:
import geopandas as gpd
import Functions
import os
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


app_path = Functions.get_input_path() / 'App'
input_path = app_path / 'Data' / 'Ground_Campaign'
ndvi = app_path / 'Documents' / 'csv' / 'NDVI'

ndvis = list(ndvi.glob('*.csv'))
ratio = pd.read_csv(app_path/'Documents'/ 'csv'/'ratio.csv', index_col=0)
print(ndvis)

In [ ]:
for file in ndvis:
 
     file_stem = Path(file).stem
     div = ratio[ratio['Name'] == file_stem]

In [ ]:
num_file = len(ndvis)
colonne = 3  
righe = int(np.ceil(num_file / colonne)) 

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(
    righe, colonne, figsize=(5 * colonne, 4 * righe), sharey=True)
axes_piatti = axes.flatten()

for i, file in enumerate(ndvis):
    nome_file = os.path.basename(file).replace(".csv", "")

    df = pd.read_csv(file, parse_dates=True, index_col='Data')

    ax_corrente = axes_piatti[i]
    ax_corrente.plot(df.index, df['NDVI_Medio'], color="steelblue", linewidth=1.5)
    ax_corrente.set_title(nome_file, fontsize=11, fontweight="bold")
    ax_corrente.tick_params(axis="x", rotation=45)

for j in range(num_file, len(axes_piatti)):
    fig.delaxes(axes_piatti[j])
    
plt.tight_layout()